# Checking the toxicity of political comments

In [1]:
!pip install detoxify pandas torch
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 63.2 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 90.3 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 97.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 81.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 91.5 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 94.3 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 85.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 89.9 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 87.0 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/

In [7]:
import pandas as pd
from detoxify import Detoxify
from tqdm import tqdm
from transformers import pipeline

In [3]:
df = pd.read_csv("/home/onyxia/work/Reddit-Polarization/annotations/toxicity/data_1_political_final.csv")

In [4]:
df

,id,user,date_ins,type,date,text,user_inter,pfp_inter,text_inter,post_title,post_text,after,language,data_1_clean_political
0,1X000149,Yare_Yare_Daze-I,NaN,comment,2020-06-01 11:49:15+00:00,LawlThe ss is bad communauty don't agree with U,Sightshade,NaN,post body,A much-needed reminder to be civil and respect...,It’s been a few days since the Origami King re...,t1_fsinitc,en,1
1,1X000757,topiga,NaN,comment,2026-01-30 09:48:26+00:00,Fin des partiels pour ma copine. Elle a changé...,AutoModerator,NaN,post body,Forum Libre - 2026-01-30,Partagez ici tout ce que vous voulez sauf la p...,t1_ny7xgij,fr,1
2,1X0007228,topiga,NaN,comment,2025-11-05 09:45:51+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1
3,1X0007231,topiga,NaN,comment,2025-11-05 08:38:57+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1
4,1X0007255,topiga,NaN,comment,2025-10-23 05:43:33+00:00,I don’t agree with everything. It’s nice to kn...,[deleted],NaN,post body,[deleted by user],[removed],t3_1nihtpj,en,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37102,1X18298,Osarel,NaN,comment,2025-09-08 15:32:18+00:00,Euh ça va ? C'est le 3ème sujet sur la remise ...,99ShahedOfBakuOfNine,NaN,post body,Le droit de vote devrait sauter pour ceux qui ...,Tout est dans le titre : le groupe minoritaire...,End,fr,1
37103,1X182912,Osarel,NaN,comment,2025-09-01 13:04:13+00:00,Pour avoir eu un père alcoolique et une mère t...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1
37104,1X182914,Osarel,NaN,comment,2025-07-18 07:26:55+00:00,Bof. Tu retiens surtout la douleur et ensuite ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1
37105,1X182915,Osarel,NaN,comment,2025-06-21 13:50:41+00:00,C'est effectivement impopulaire je pense. Moi ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1


In [4]:
model = Detoxify(
    'multilingual',
    device='cpu'
)

batch_size = 64

toxicity_scores = []

comments = df["text"].fillna("").tolist()

for i in tqdm(range(0, len(comments), batch_size)):
    batch = comments[i:i+batch_size]

    results = model.predict(batch)

    toxicity_scores.extend(results["toxicity"])

df["toxicity"] = toxicity_scores

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.4-alpha/multilingual_debiased-0b549669.ckpt" to /home/onyxia/.cache/torch/hub/checkpoints/multilingual_debiased-0b549669.ckpt


100%|██████████| 1.04G/1.04G [00:10<00:00, 102MB/s] 
100%|██████████| 580/580 [1:42:06<00:00, 10.56s/it]


In [6]:
df.to_csv("data_1_toxicity.csv", index=False, encoding="utf-8")

In [19]:
df

,id,user,date_ins,type,date,text,user_inter,pfp_inter,text_inter,post_title,post_text,after,language,data_1_clean_political,toxicity
0,1X000149,Yare_Yare_Daze-I,NaN,comment,2020-06-01 11:49:15+00:00,LawlThe ss is bad communauty don't agree with U,Sightshade,NaN,post body,A much-needed reminder to be civil and respect...,It’s been a few days since the Origami King re...,t1_fsinitc,en,1,0.753615
1,1X000757,topiga,NaN,comment,2026-01-30 09:48:26+00:00,Fin des partiels pour ma copine. Elle a changé...,AutoModerator,NaN,post body,Forum Libre - 2026-01-30,Partagez ici tout ce que vous voulez sauf la p...,t1_ny7xgij,fr,1,0.001628
2,1X0007228,topiga,NaN,comment,2025-11-05 09:45:51+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1,0.000452
3,1X0007231,topiga,NaN,comment,2025-11-05 08:38:57+00:00,EU 30,qidi_3dprinter,NaN,post body,🎃 QIDI Halloween Treasure Hunt Giveaway! 👻,It’s time for some fun! Many pumpkins are wait...,t1_nlsoxsz,en,1,0.000452
4,1X0007255,topiga,NaN,comment,2025-10-23 05:43:33+00:00,I don’t agree with everything. It’s nice to kn...,[deleted],NaN,post body,[deleted by user],[removed],t3_1nihtpj,en,1,0.010784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37102,1X18298,Osarel,NaN,comment,2025-09-08 15:32:18+00:00,Euh ça va ? C'est le 3ème sujet sur la remise ...,99ShahedOfBakuOfNine,NaN,post body,Le droit de vote devrait sauter pour ceux qui ...,Tout est dans le titre : le groupe minoritaire...,End,fr,1,0.000552
37103,1X182912,Osarel,NaN,comment,2025-09-01 13:04:13+00:00,Pour avoir eu un père alcoolique et une mère t...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1,0.188919
37104,1X182914,Osarel,NaN,comment,2025-07-18 07:26:55+00:00,Bof. Tu retiens surtout la douleur et ensuite ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1,0.301428
37105,1X182915,Osarel,NaN,comment,2025-06-21 13:50:41+00:00,C'est effectivement impopulaire je pense. Moi ...,[deleted],NaN,post body,[deleted by user],[removed],End,fr,1,0.007133


## Hugging Face CitizenLab

In [8]:
df_2 = pd.read_csv("/home/onyxia/work/Reddit-Polarization/annotations/toxicity/data_2_political_final.csv")

In [9]:
classifier = pipeline(
    "text-classification",
    model="citizenlab/distilbert-base-multilingual-cased-toxicity",
    batch_size=64,
    device=-1
)

comments = df_2["text"].fillna("").tolist()

results = []

for i in tqdm(range(0, len(comments), 64)):
    batch = comments[i:i+64]

    preds = classifier(
        batch,
        truncation=True,
        max_length=256
    )

    results.extend(preds)

df_2["toxicity_score"] = [x["score"] for x in results]
df_2["toxicity_label"] = [x["label"] for x in results]

100%|██████████| 562/562 [31:49<00:00,  3.40s/it]


In [11]:
df_2.to_csv("data_2_toxicity.csv", index=False, encoding="utf-8")

In [12]:
df_1 = pd.read_csv("/home/onyxia/work/Reddit-Polarization/annotations/toxicity/data_1_political_final.csv")

In [13]:
classifier = pipeline(
    "text-classification",
    model="citizenlab/distilbert-base-multilingual-cased-toxicity",
    batch_size=64,
    device=-1
)

comments = df_1["text"].fillna("").tolist()

results = []

for i in tqdm(range(0, len(comments), 64)):
    batch = comments[i:i+64]

    preds = classifier(
        batch,
        truncation=True,
        max_length=256
    )

    results.extend(preds)

df_1["toxicity_score"] = [x["score"] for x in results]
df_1["toxicity_label"] = [x["label"] for x in results]


df_1.to_csv("data_1_toxicity_hf.csv", index=False, encoding="utf-8")

100%|██████████| 580/580 [33:49<00:00,  3.50s/it]


In [18]:
df_2["toxicity_label"].value_counts()

toxicity_label
not_toxic    33952
toxic         1978
Name: count, dtype: int64